# Data Cleaning and Preprocessing

Este notebook prepara el dataset del experimento para analisis posterior. El flujo incluye:

- carga del archivo fuente,
- limpieza y estandarizacion de columnas,
- revision basica de calidad de datos,
- completado de variables grupales por `team_id`,
- transformacion de columnas multivalor a listas,
- conversion de respuestas Likert a escala numerica,
- construccion de datasets final, individual y por equipo,
- exportacion en archivos Excel.

## 1. Imports

In [279]:
from pathlib import Path

import pandas as pd
import re
import unicodedata


## 2. Carga del dataset

In [280]:
candidate_paths = [
    Path.cwd().resolve().parent / "Data" / "Data.xlsx",
    Path.cwd().resolve() / "Data" / "Data.xlsx",
]

path = next((candidate for candidate in candidate_paths if candidate.exists()), None)
if path is None:
    raise FileNotFoundError("No se encontro Data/Data.xlsx desde el directorio actual.")

path


WindowsPath('C:/Users/Franc/OneDrive/Documentos/diversity-experiment-repo/Data/Data.xlsx')

In [281]:
dataset = pd.read_excel(path, sheet_name="Data")
dataset.head()


,Team_ID,Usuario,Participant_ID,Diversity,Mission,Areas_individual,Areas_grupal,Ideas_individual,N° ideas_individual,Ideas_team,...,Q11,Q12,Q13,Q14,Q15,Q16,Q17,Q18,Unnamed: 29,Unnamed: 30
0,T1,06p068,1,1,1,"Salud y bienestar, Ambiente y acción climática...","Salud y bienestar, Educación y conocimiento, I...","Mejores ofertas laborales, de acuerdo a los tí...",5,Seguridad ciudadana. / Movilidad eficiente. / ...,...,Nunca,Nunca,Muy frecuentemente,Frecuentemente,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,NaN,NaN
1,T1,09701C,2,1,1,"Educación y conocimiento, Ambiente y acción c...",NaN,"Transporte asequible, seguridad, salud, equ...",1,NaN,...,Nunca,Nunca,Frecuentemente,Frecuentemente,Frecuentemente,Frecuentemente,Frecuentemente,Frecuentemente,NaN,NaN
2,T1,70G9NP,3,1,1,"Salud y bienestar, Educación y conocimiento, I...",NaN,Movilidad eficiente / Mayor cobertura de servi...,4,NaN,...,Nunca,Nunca,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,NaN,NaN
3,T2,43010Q,1,1,1,"Infraestructura sustentable, Salud y bienestar...","Inclusión social y equidad, Educación y conoc...",Apoyo médico para las personas más vulnerables...,6,Canalización eficiente de recursos a la educac...,...,Nunca,Nunca,Nunca,Muy frecuentemente,Frecuentemente,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,NaN
4,T2,689793,2,1,1,"Educación y conocimiento, Salud y bienestar, A...",NaN,-Educacion Ambiental: desde los colegios los p...,3,NaN,...,Nunca,Nunca,Nunca,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,Muy frecuentemente,NaN


## 3. Limpieza y estandarizacion inicial

In [282]:
# Normalizar nombres de columnas y renombrar columnas unnamed siguiendo la ultima columna numerada.
dataset.columns = (
    dataset.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace(r"[^0-9a-z_]", "", regex=True)
)

new_columns = []
last_prefix = None
last_number = None

for col in dataset.columns:
    if "unnamed" in col:
        if last_prefix is None or last_number is None:
            raise ValueError(f"No se puede inferir el nombre para la columna {col!r}.")
        last_number += 1
        new_columns.append(f"{last_prefix}{last_number}")
        continue

    new_columns.append(col)
    match = re.fullmatch(r"([a-z_]+?)(\d+)", col)
    if match:
        last_prefix = match.group(1)
        last_number = int(match.group(2))

dataset.columns = new_columns

# Estandarizar identificadores de usuario.
dataset["usuario"] = dataset["usuario"].astype(str).str.strip().str.upper()

dataset.columns


Index(['team_id', 'usuario', 'participant_id', 'diversity', 'mission',
       'areas_individual', 'areas_grupal', 'ideas_individual',
       'n_ideas_individual', 'ideas_team', 'n_ideas_team', 'q1', 'q2', 'q3',
       'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14',
       'q15', 'q16', 'q17', 'q18', 'q19', 'q20'],
      dtype='str')

## 4. Revision de calidad de datos

In [283]:
# Ajustar tipos y crear una copia de trabajo.
dataset["n_ideas_team"] = dataset["n_ideas_team"].astype("Int64")
dataset_clean = dataset.copy()

# Variables grupales registradas por un solo integrante se propagan al resto del equipo.
group_columns = ["areas_grupal", "ideas_team", "n_ideas_team"]
dataset_clean[group_columns] = (
    dataset_clean.groupby("team_id")[group_columns]
    .transform(lambda group: group.ffill().bfill())
)

dataset_clean.info()


<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   team_id             240 non-null    str  
 1   usuario             240 non-null    str  
 2   participant_id      240 non-null    int64
 3   diversity           240 non-null    int64
 4   mission             240 non-null    int64
 5   areas_individual    240 non-null    str  
 6   areas_grupal        240 non-null    str  
 7   ideas_individual    238 non-null    str  
 8   n_ideas_individual  240 non-null    int64
 9   ideas_team          240 non-null    str  
 10  n_ideas_team        240 non-null    Int64
 11  q1                  240 non-null    str  
 12  q2                  240 non-null    str  
 13  q3                  240 non-null    str  
 14  q4                  240 non-null    str  
 15  q5                  240 non-null    str  
 16  q6                  240 non-null    str  
 17  q7      

In [284]:
null_summary = (
    dataset_clean.isna()
    .sum()
    .rename("null_count")
    .to_frame()
    .assign(null_pct=lambda df: (df["null_count"] / len(dataset_clean) * 100).round(2))
    .sort_values(["null_count", "null_pct"], ascending=False)
)

null_summary[null_summary["null_count"] > 0]


,null_count,null_pct
q20,177,73.75
q19,3,1.25
ideas_individual,2,0.83


In [285]:
duplicate_summary = pd.Series(
    {
        "duplicated_usuario": int(dataset_clean["usuario"].duplicated().sum()),
        "duplicated_team_participant": int(dataset_clean.duplicated(subset=["team_id", "participant_id"]).sum()),
    },
    name="duplicate_count",
)

duplicate_summary


duplicated_usuario             3
duplicated_team_participant    0
Name: duplicate_count, dtype: int64

## 5. Funciones auxiliares

In [286]:
def clean_area_item(item):
    """Limpia prefijos y espacios en los campos de areas."""
    item = str(item).strip()
    item = unicodedata.normalize("NFKC", item)
    item = re.sub(r"^[XV]\s*-\s*", "", item, flags=re.IGNORECASE)
    item = re.sub(r"\s+", " ", item)
    return item.strip()


def limpiar_texto(texto):
    """Normaliza saltos de linea y espacios para las columnas de ideas."""
    if pd.isna(texto):
        return ""

    texto = str(texto)
    texto = texto.replace("\n", " / ")
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


def separar_ideas_seguro(texto):
    """Separa ideas sin romper frases largas con comas internas."""
    texto = limpiar_texto(texto)
    if texto == "":
        return []

    ideas = []
    partes = re.split(r"\s*/\s*", texto) if "/" in texto else [texto]

    for parte in partes:
        parte = parte.strip()
        subpartes = re.split(r"\s*(?:\d+\.\s+)", parte)

        for sub in subpartes:
            sub = sub.strip(" -.;:")
            if sub == "":
                continue

            coma_partes = [p.strip() for p in sub.split(",")]
            if len(coma_partes) >= 3:
                promedio_palabras = sum(len(p.split()) for p in coma_partes) / len(coma_partes)
                if promedio_palabras <= 4:
                    ideas.extend(coma_partes)
                else:
                    ideas.append(sub)
            else:
                ideas.append(sub)

    resultado = []
    for idea in ideas:
        idea = idea.strip(" -.;:")
        idea = re.sub(r"\s+", " ", idea)
        if len(idea) > 2 and idea.lower() not in [x.lower() for x in resultado]:
            resultado.append(idea)

    return resultado


def area_slug(area):
    """Convierte el nombre del area en un slug apto para columnas."""
    value = unicodedata.normalize("NFKD", area.lower())
    value = "".join(char for char in value if not unicodedata.combining(char))
    value = re.sub(r"\s+", "_", value)
    return value


def clean_likert_response(value):
    """Toma la primera respuesta si hay varias y convierte '/' en NA."""
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()
    if value == "/":
        return pd.NA
    if "," in value:
        return value.split(",")[0].strip()
    return value


## 6. Transformacion de areas

In [287]:
dataset_clean["areas_individual_list"] = (
    dataset_clean["areas_individual"]
    .fillna("")
    .astype(str)
    .str.split(",")
    .apply(lambda items: [clean_area_item(item) for item in items if clean_area_item(item)])
)

dataset_clean["areas_grupal_list"] = (
    dataset_clean["areas_grupal"]
    .fillna("")
    .astype(str)
    .str.split(",")
    .apply(lambda items: [clean_area_item(item) for item in items if clean_area_item(item)])
)

dataset_clean[["areas_individual_list", "areas_grupal_list"]].head()


,areas_individual_list,areas_grupal_list
0,"[Salud y bienestar, Ambiente y acción climátic...","[Salud y bienestar, Educación y conocimiento, ..."
1,"[Educación y conocimiento, Ambiente y acción c...","[Salud y bienestar, Educación y conocimiento, ..."
2,"[Salud y bienestar, Educación y conocimiento, ...","[Salud y bienestar, Educación y conocimiento, ..."
3,"[Infraestructura sustentable, Salud y bienesta...","[Inclusión social y equidad, Educación y conoc..."
4,"[Educación y conocimiento, Salud y bienestar, ...","[Inclusión social y equidad, Educación y conoc..."


In [288]:
# Crear columnas de ranking para areas individuales y grupales.
all_areas = sorted(
    {area for areas in dataset_clean["areas_individual_list"] for area in areas}
    | {area for areas in dataset_clean["areas_grupal_list"] for area in areas}
)

for area in all_areas:
    dataset_clean[f"areas_individual__{area_slug(area)}"] = dataset_clean["areas_individual_list"].apply(
        lambda areas: areas.index(area) + 1 if area in areas else pd.NA
    ).astype("Int64")

    dataset_clean[f"areas_grupal__{area_slug(area)}"] = dataset_clean["areas_grupal_list"].apply(
        lambda areas: areas.index(area) + 1 if area in areas else pd.NA
    ).astype("Int64")

dataset_clean[[
    "areas_individual_list",
    "areas_grupal_list",
    *[col for col in dataset_clean.columns if col.startswith("areas_individual__")][:6],
    *[col for col in dataset_clean.columns if col.startswith("areas_grupal__")][:6],
]].head()


,areas_individual_list,areas_grupal_list,areas_individual__ambiente_y_accion_climatica,areas_individual__educacion_y_conocimiento,areas_individual__inclusion_social_y_equidad,areas_individual__infraestructura_sustentable,areas_individual__salud_y_bienestar,areas_individual__trabajo_decente_y_desarrollo_economico,areas_grupal__ambiente_y_accion_climatica,areas_grupal__educacion_y_conocimiento,areas_grupal__inclusion_social_y_equidad,areas_grupal__infraestructura_sustentable,areas_grupal__salud_y_bienestar,areas_grupal__trabajo_decente_y_desarrollo_economico
0,"[Salud y bienestar, Ambiente y acción climátic...","[Salud y bienestar, Educación y conocimiento, ...",2,3,5,6,1,4,5,2,3,6,1,4
1,"[Educación y conocimiento, Ambiente y acción c...","[Salud y bienestar, Educación y conocimiento, ...",2,1,6,4,3,5,5,2,3,6,1,4
2,"[Salud y bienestar, Educación y conocimiento, ...","[Salud y bienestar, Educación y conocimiento, ...",6,2,3,5,1,4,5,2,3,6,1,4
3,"[Infraestructura sustentable, Salud y bienesta...","[Inclusión social y equidad, Educación y conoc...",3,5,6,1,2,4,6,2,1,5,3,4
4,"[Educación y conocimiento, Salud y bienestar, ...","[Inclusión social y equidad, Educación y conoc...",3,1,6,5,2,4,6,2,1,5,3,4


## 7. Transformacion de ideas

In [289]:
dataset_clean["ideas_individual_list"] = dataset_clean["ideas_individual"].apply(separar_ideas_seguro)
dataset_clean["ideas_team_list"] = dataset_clean["ideas_team"].apply(separar_ideas_seguro)

dataset_clean[["ideas_individual_list", "ideas_team_list"]].head()


,ideas_individual_list,ideas_team_list
0,"[Mejores ofertas laborales, de acuerdo a los t...","[Seguridad ciudadana, Movilidad eficiente, Mej..."
1,"[Transporte asequible, seguridad, salud, equid...","[Seguridad ciudadana, Movilidad eficiente, Mej..."
2,"[Movilidad eficiente, Mayor cobertura de servi...","[Seguridad ciudadana, Movilidad eficiente, Mej..."
3,[Apoyo médico para las personas más vulnerable...,[Canalización eficiente de recursos a la educa...
4,[Educacion Ambiental: desde los colegios los p...,[Canalización eficiente de recursos a la educa...


## 8. Conversion de escala Likert

In [290]:
likert_mapping = {
    "Nunca": 1,
    "Rara vez": 2,
    "Algunas veces": 3,
    "Frecuentemente": 4,
    "Muy frecuentemente": 5,
}

likert_columns = [f"q{i}" for i in range(1, 21)]

for column in likert_columns:
    dataset_clean[column] = dataset_clean[column].apply(clean_likert_response)
    dataset_clean[f"{column}_num"] = dataset_clean[column].map(likert_mapping).astype("Int64")

dataset_clean[["q1", "q1_num", "q4", "q4_num", "q5", "q5_num"]].head()


,q1,q1_num,q4,q4_num,q5,q5_num
0,Algunas veces,3,Muy frecuentemente,5,Muy frecuentemente,5
1,Muy frecuentemente,5,Muy frecuentemente,5,Frecuentemente,4
2,Frecuentemente,4,Muy frecuentemente,5,Frecuentemente,4
3,Nunca,1,Frecuentemente,4,Muy frecuentemente,5
4,Muy frecuentemente,5,Frecuentemente,4,Frecuentemente,4


## 9. Seleccion final de columnas

In [291]:
# Eliminar columnas textuales originales ya transformadas.
dataset_clean = dataset_clean.drop(
    columns=["areas_individual", "areas_grupal", "ideas_individual", "ideas_team"]
)

dataset_clean[[
    "team_id",
    "usuario",
    "areas_individual_list",
    "areas_grupal_list",
    "ideas_individual_list",
    "ideas_team_list",
    *[col for col in dataset_clean.columns if col.startswith("areas_individual__")][:6],
    *[col for col in dataset_clean.columns if col.startswith("areas_grupal__")][:6],
    "q1_num",
    "q2_num",
    "q3_num",
]].head()


,team_id,usuario,areas_individual_list,areas_grupal_list,ideas_individual_list,ideas_team_list,areas_individual__ambiente_y_accion_climatica,areas_individual__educacion_y_conocimiento,areas_individual__inclusion_social_y_equidad,areas_individual__infraestructura_sustentable,...,areas_individual__trabajo_decente_y_desarrollo_economico,areas_grupal__ambiente_y_accion_climatica,areas_grupal__educacion_y_conocimiento,areas_grupal__inclusion_social_y_equidad,areas_grupal__infraestructura_sustentable,areas_grupal__salud_y_bienestar,areas_grupal__trabajo_decente_y_desarrollo_economico,q1_num,q2_num,q3_num
0,T1,06P068,"[Salud y bienestar, Ambiente y acción climátic...","[Salud y bienestar, Educación y conocimiento, ...","[Mejores ofertas laborales, de acuerdo a los t...","[Seguridad ciudadana, Movilidad eficiente, Mej...",2,3,5,6,...,4,5,2,3,6,1,4,3,5,5
1,T1,09701C,"[Educación y conocimiento, Ambiente y acción c...","[Salud y bienestar, Educación y conocimiento, ...","[Transporte asequible, seguridad, salud, equid...","[Seguridad ciudadana, Movilidad eficiente, Mej...",2,1,6,4,...,5,5,2,3,6,1,4,5,4,4
2,T1,70G9NP,"[Salud y bienestar, Educación y conocimiento, ...","[Salud y bienestar, Educación y conocimiento, ...","[Movilidad eficiente, Mayor cobertura de servi...","[Seguridad ciudadana, Movilidad eficiente, Mej...",6,2,3,5,...,4,5,2,3,6,1,4,4,5,5
3,T2,43010Q,"[Infraestructura sustentable, Salud y bienesta...","[Inclusión social y equidad, Educación y conoc...",[Apoyo médico para las personas más vulnerable...,[Canalización eficiente de recursos a la educa...,3,5,6,1,...,4,6,2,1,5,3,4,1,1,1
4,T2,689793,"[Educación y conocimiento, Salud y bienestar, ...","[Inclusión social y equidad, Educación y conoc...",[Educacion Ambiental: desde los colegios los p...,[Canalización eficiente de recursos a la educa...,3,1,6,5,...,4,6,2,1,5,3,4,5,2,5


## 10. Construccion de datasets finales

In [292]:
# Dataset principal: version completa y limpia.
dataset_principal = dataset_clean.copy()

# Dataset individual: una fila por participante.
individual_columns = [
    "team_id",
    "usuario",
    "participant_id",
    "diversity",
    "mission",
    "areas_individual_list",
    "areas_grupal_list",
    "ideas_individual_list",
    "ideas_team_list",
    "n_ideas_individual",
    "n_ideas_team",
] + [col for col in dataset_principal.columns if col.startswith("areas_individual__")] + [col for col in dataset_principal.columns if col.startswith("areas_grupal__")] + [col for col in dataset_principal.columns if re.fullmatch(r"q\d+_num", col)]

dataset_individual = dataset_principal[individual_columns].copy()

# Dataset equipo: una fila por team_id con agregacion de respuestas numericas.
team_aggregations = {
    "diversity": "first",
    "mission": "first",
    "n_ideas_team": "first",
    "areas_grupal_list": "first",
    "ideas_team_list": "first",
    "participant_id": "count",
}

for col in [c for c in dataset_principal.columns if c.startswith("areas_grupal__")]:
    team_aggregations[col] = "first"

for col in [c for c in dataset_principal.columns if re.fullmatch(r"q\d+_num", c)]:
    team_aggregations[col] = "mean"

dataset_team = (
    dataset_principal.groupby("team_id", as_index=False)
    .agg(team_aggregations)
    .rename(columns={"participant_id": "team_size"})
)

dataset_principal.shape, dataset_individual.shape, dataset_team.shape


((240, 63), (240, 43), (80, 33))

## 11. Exportacion

In [293]:
output_dir = path.parent
output_dir.mkdir(parents=True, exist_ok=True)

principal_path = output_dir / "Data_clean_final.xlsx"
individual_path = output_dir / "Data_clean_individual.xlsx"
team_path = output_dir / "Data_clean_team.xlsx"

dataset_principal.to_excel(principal_path, index=False)
dataset_individual.to_excel(individual_path, index=False)
dataset_team.to_excel(team_path, index=False)

{
    "dataset_principal": principal_path,
    "dataset_individual": individual_path,
    "dataset_team": team_path,
}


{'dataset_principal': WindowsPath('C:/Users/Franc/OneDrive/Documentos/diversity-experiment-repo/Data/Data_clean_final.xlsx'),
 'dataset_individual': WindowsPath('C:/Users/Franc/OneDrive/Documentos/diversity-experiment-repo/Data/Data_clean_individual.xlsx'),
 'dataset_team': WindowsPath('C:/Users/Franc/OneDrive/Documentos/diversity-experiment-repo/Data/Data_clean_team.xlsx')}